In [1]:
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

GPU: Tesla T4
CUDA: True


In [2]:
import torch, torch.nn as nn
import torchvision, torchvision.transforms as T

# load digits dataset (built-in, free)
train_set = torchvision.datasets.MNIST('/content', train=True, download=True,
              transform=T.ToTensor())
loader = torch.utils.data.DataLoader(train_set, batch_size=256, shuffle=True)

# model ON the gpu
model = nn.Sequential(nn.Flatten(), nn.Linear(784,256), nn.ReLU(),
                      nn.Linear(256,10)).cuda()
opt = torch.optim.Adam(model.parameters())
lossf = nn.CrossEntropyLoss()

for epoch in range(3):
    for x,y in loader:
        x,y = x.view(-1,784).cuda(), y.cuda()
        opt.zero_grad()
        loss = lossf(model(x), y)
        loss.backward(); opt.step()
    print(f"epoch {epoch}: loss {loss.item():.4f}")

# test accuracy
model.eval(); correct=0
with torch.no_grad():
    for x,y in torch.utils.data.DataLoader(
        torchvision.datasets.MNIST('/content',train=False,transform=T.ToTensor()),256):
        correct += (model(x.view(-1,784).cuda()).argmax(1)==y.cuda()).sum().item()
print("TEST ACC:", correct/10000)

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 494kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.59MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.7MB/s]


epoch 0: loss 0.1427
epoch 1: loss 0.1095
epoch 2: loss 0.0611
TEST ACC: 0.963


In [3]:
import torch, torch.nn as nn, torchvision, torchvision.transforms as T

# rebuild + load trained weights? (quick: retrain 1 epoch for demo)
train_set = torchvision.datasets.MNIST('/content', train=True, download=True, transform=T.ToTensor())
loader = torch.utils.data.DataLoader(train_set, batch_size=256, shuffle=True)
model = nn.Sequential(nn.Flatten(), nn.Linear(784,256), nn.ReLU(), nn.Linear(256,10)).cuda()
opt = torch.optim.Adam(model.parameters()); lossf = nn.CrossEntropyLoss()
for x,y in loader:
    x,y=x.view(-1,784).cuda(),y.cuda(); opt.zero_grad()
    lossf(model(x),y).backward(); opt.step()

# FGSM attack
eps = 0.3
x,y = next(iter(torch.utils.data.DataLoader(
    torchvision.datasets.MNIST('/content',train=False,transform=T.ToTensor()),256)))
x,y = x.view(-1,784).cuda(), y.cuda()
x.requires_grad_(True)
loss = lossf(model(x), y)
loss.backward()
adv = x + eps*x.grad.sign()         # the attack: tiny noise
adv = adv.clamp(0,1).detach()

clean = (model(x).argmax(1)==y).float().mean().item()
attacked = (model(adv).argmax(1)==y).float().mean().item()
print(f"clean acc: {clean:.3f}  ->  under attack: {attacked:.3f}")

clean acc: 0.949  ->  under attack: 0.000


In [7]:
def acc(model, loader, attack=False):
    model.eval(); c=n=0
    for x,y in loader:
        x,y = x.view(-1,784).cuda(), y.cuda()
        if attack:
            model.train()
            x2 = x.clone().requires_grad_(True)
            lossf(model(x2), y).backward()
            x = (x2 + eps*x2.grad.sign()).clamp(0,1).detach()
            model.eval()
        with torch.no_grad():
            c += (model(x).argmax(1)==y).sum().item(); n += y.size(0)
    return c/n

In [8]:
print(f"clean acc:    {acc(model, test_loader):.3f}")
print(f"attack acc:   {acc(model, test_loader, attack=True):.3f}")

clean acc:    0.961
attack acc:   0.711
